# UCB Supply Chain — Forecasting Pipeline (All Drugs)
### Section 1 — Install & Import

In [ ]:
!pip install pandas numpy matplotlib seaborn prophet scikit-learn boto3 -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import json
import os
import math
import boto3
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")
matplotlib.rcParams['figure.figsize'] = (14, 5)
pd.set_option('display.float_format', '{:,.1f}'.format)

print("Done.")

---
### Section 2 — Load Medicare Part D demand data

In [ ]:
df_demand   = pd.read_csv("aed_demand_by_state.csv")
df_national = pd.read_csv("aed_demand_national.csv")

print(f"Demand by state : {df_demand.shape}")
print(f"National totals : {df_national.shape}")
print()
print("Drugs available:")
for d in sorted(df_demand["drug"].unique()):
    total = df_demand[df_demand["drug"]==d]["total_claims"].sum()
    print(f"  {d:<20} {total:>12,.0f} total claims")

---
### Section 3 — Convert annual data to weekly time series

NOTE: Weekly demand is synthetically generated from real 2022 Medicare Part D
annual baselines. In production this would be replaced by UCB's internal
weekly sales and order data.

In [ ]:
def annual_to_weekly(drug_name, state, annual_claims, year=2022, seed=42):
    """
    Convert annual claim count to a 52-week time series.
    Adds seasonal variation and realistic noise.
    NOTE: Synthetic weekly distribution — real annual baseline.
    """
    np.random.seed(seed)
    weeks = pd.date_range(start=f"{year}-01-01", periods=52, freq="W")

    # Seasonal weights — slightly higher in winter
    month_weights = {1:1.08, 2:1.06, 3:1.02, 4:0.98, 5:0.96, 6:0.94,
                     7:0.93, 8:0.95, 9:0.99, 10:1.02, 11:1.05, 12:1.07}
    seasonal = np.array([month_weights[w.month] for w in weeks])

    base_weekly  = annual_claims / 52
    noise        = np.random.normal(1.0, 0.08, 52)
    noise        = np.clip(noise, 0.8, 1.2)
    weekly_demand = np.round(base_weekly * seasonal * noise).astype(int)
    weekly_demand = np.clip(weekly_demand, 0, None)

    return pd.DataFrame({
        "date"  : weeks,
        "drug"  : drug_name,
        "state" : state,
        "claims": weekly_demand,
    })

# Build weekly time series for all drugs and states
print("Converting annual to weekly demand (synthetic distribution, real baseline)...")
weekly_series = []

for _, row in df_demand.iterrows():
    if row["total_claims"] > 0:
        series = annual_to_weekly(
            drug_name    = row["drug"],
            state        = row["state"],
            annual_claims= row["total_claims"],
            seed         = hash(f"{row['drug']}_{row['state']}") % 10000
        )
        weekly_series.append(series)

df_weekly = pd.concat(weekly_series, ignore_index=True)
print(f"Weekly series shape : {df_weekly.shape}")
print(f"Date range          : {df_weekly['date'].min()} to {df_weekly['date'].max()}")
print()
print(df_weekly.head(8))

In [ ]:
# National weekly aggregate
df_national_weekly = (
    df_weekly.groupby(["date", "drug"])["claims"]
    .sum()
    .reset_index()
)

print(f"National weekly shape: {df_national_weekly.shape}")

---
### Section 4 — Visualize baseline demand for all drugs

In [ ]:
all_drugs = sorted(df_national_weekly["drug"].unique())
cols      = 2
rows      = math.ceil(len(all_drugs) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
axes_flat = axes.flatten()

colors = ["#2E5F8A", "#1D9E75", "#EF9F27", "#E24B4A",
          "#7F77DD", "#085041", "#712B13", "#0C447C",
          "#444441", "#633806"]

for i, drug in enumerate(all_drugs):
    ax   = axes_flat[i]
    data = df_national_weekly[df_national_weekly["drug"] == drug]
    ax.plot(data["date"], data["claims"], color=colors[i % len(colors)], linewidth=1.5)
    ax.fill_between(data["date"], data["claims"], alpha=0.12, color=colors[i % len(colors)])
    ax.set_title(f"{drug}", fontsize=11)
    ax.set_xlabel("Week")
    ax.set_ylabel("Claims")
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for j in range(len(all_drugs), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("National Weekly Demand Baseline — All AEDs (2022, Synthetic Weekly Distribution)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("all_drugs_weekly_baseline.png", dpi=150)
plt.show()

---
### Section 5 — Inject anomalies across all drugs

Simulating real-world supply chain events:
- Demand spikes (competitor shortage, formulary win)
- Demand drops (distribution disruption, formulary change)

In [ ]:
ANOMALY_EVENTS = [
    # UCB drugs
    {
        "name"      : "Competitor shortage — Levetiracetam Florida",
        "drug"      : "Levetiracetam",
        "state"     : "FL",
        "week_start": "2022-06-01",
        "week_end"  : "2022-07-01",
        "multiplier": 3.2,
        "type"      : "spike",
    },
    {
        "name"      : "Formulary change — Levetiracetam New York",
        "drug"      : "Levetiracetam",
        "state"     : "NY",
        "week_start": "2022-09-01",
        "week_end"  : "2022-10-01",
        "multiplier": 0.3,
        "type"      : "drop",
    },
    {
        "name"      : "Formulary win — Brivaracetam California",
        "drug"      : "Brivaracetam",
        "state"     : "CA",
        "week_start": "2022-04-01",
        "week_end"  : "2022-05-01",
        "multiplier": 2.5,
        "type"      : "spike",
    },
    {
        "name"      : "Distribution disruption — Brivaracetam Texas",
        "drug"      : "Brivaracetam",
        "state"     : "TX",
        "week_start": "2022-11-01",
        "week_end"  : "2022-11-15",
        "multiplier": 0.1,
        "type"      : "drop",
    },
    # Competitive AEDs
    {
        "name"      : "Supply disruption — Lamotrigine Texas",
        "drug"      : "Lamotrigine",
        "state"     : "TX",
        "week_start": "2022-08-01",
        "week_end"  : "2022-09-01",
        "multiplier": 0.2,
        "type"      : "drop",
    },
    {
        "name"      : "Demand spike — Topiramate Florida",
        "drug"      : "Topiramate",
        "state"     : "FL",
        "week_start": "2022-05-01",
        "week_end"  : "2022-06-01",
        "multiplier": 2.8,
        "type"      : "spike",
    },
    {
        "name"      : "Distribution issue — Carbamazepine Ohio",
        "drug"      : "Carbamazepine",
        "state"     : "OH",
        "week_start": "2022-10-01",
        "week_end"  : "2022-10-15",
        "multiplier": 0.1,
        "type"      : "drop",
    },
    {
        "name"      : "Formulary win — Lacosamide California",
        "drug"      : "Lacosamide",
        "state"     : "CA",
        "week_start": "2022-03-01",
        "week_end"  : "2022-04-01",
        "multiplier": 2.2,
        "type"      : "spike",
    },
    {
        "name"      : "Demand drop — Valproic Acid Florida",
        "drug"      : "Valproic Acid",
        "state"     : "FL",
        "week_start": "2022-07-01",
        "week_end"  : "2022-08-01",
        "multiplier": 0.3,
        "type"      : "drop",
    },
    {
        "name"      : "Shortage — Zonisamide California",
        "drug"      : "Zonisamide",
        "state"     : "CA",
        "week_start": "2022-09-01",
        "week_end"  : "2022-10-01",
        "multiplier": 0.15,
        "type"      : "drop",
    },
]

# Inject anomalies
df_anomaly = df_weekly.copy()
df_anomaly["is_anomaly"]   = False
df_anomaly["anomaly_name"] = ""

for event in ANOMALY_EVENTS:
    mask = (
        (df_anomaly["drug"]  == event["drug"])  &
        (df_anomaly["state"] == event["state"]) &
        (df_anomaly["date"]  >= event["week_start"]) &
        (df_anomaly["date"]  <= event["week_end"])
    )
    df_anomaly.loc[mask, "claims"]       = (df_anomaly.loc[mask, "claims"] * event["multiplier"]).astype(int)
    df_anomaly.loc[mask, "is_anomaly"]   = True
    df_anomaly.loc[mask, "anomaly_name"] = event["name"]
    print(f"Injected: {event['name']} — {mask.sum()} weeks affected")

print(f"\nTotal anomaly weeks: {df_anomaly['is_anomaly'].sum()}")

In [ ]:
# Visualize all injected anomalies
n_events = len(ANOMALY_EVENTS)
cols     = 2
rows     = math.ceil(n_events / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3.5))
axes_flat = axes.flatten()

for i, event in enumerate(ANOMALY_EVENTS):
    ax      = axes_flat[i]
    drug    = event["drug"]
    state   = event["state"]
    normal  = df_weekly[(df_weekly["drug"] == drug) & (df_weekly["state"] == state)]
    anomaly = df_anomaly[(df_anomaly["drug"] == drug) & (df_anomaly["state"] == state)]

    ax.plot(normal["date"], normal["claims"],  color="#94a3b8", linewidth=1,   label="Normal", alpha=0.7)
    ax.plot(anomaly["date"], anomaly["claims"], color="#2E5F8A", linewidth=1.5, label="With anomaly")

    anom_weeks = anomaly[anomaly["is_anomaly"]]
    ax.scatter(anom_weeks["date"], anom_weeks["claims"],
               color="#E24B4A", zorder=5, s=30, label="Anomaly")

    ax.set_title(f"{event['name']}", fontsize=10)
    ax.set_xlabel("Week")
    ax.set_ylabel("Claims")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(n_events, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Injected Supply Chain Anomalies — All Drugs", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("all_drugs_injected_anomalies.png", dpi=150)
plt.show()

---
### Section 6 — Train Prophet forecasting models for all drugs

In [ ]:
from prophet import Prophet

def train_prophet(drug, state, df):
    """Train a Prophet model for one drug-state combination."""
    data = (
        df[(df["drug"] == drug) & (df["state"] == state)]
        .rename(columns={"date": "ds", "claims": "y"})
        [["ds", "y"]]
        .sort_values("ds")
    )
    if len(data) < 10:
        return None, None

    model = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        seasonality_mode        = "multiplicative",
        interval_width          = 0.95,
        changepoint_prior_scale = 0.05,
    )
    model.fit(data)
    future   = model.make_future_dataframe(periods=4, freq="W")
    forecast = model.predict(future)
    return model, forecast


# All drugs — top 3 states each by demand volume
ALL_DRUG_STATES = [
    ("Levetiracetam", "FL"),
    ("Levetiracetam", "CA"),
    ("Levetiracetam", "TX"),
    ("Brivaracetam",  "CA"),
    ("Brivaracetam",  "FL"),
    ("Brivaracetam",  "TX"),
    ("Lamotrigine",   "CA"),
    ("Lamotrigine",   "FL"),
    ("Lamotrigine",   "TX"),
    ("Topiramate",    "FL"),
    ("Topiramate",    "CA"),
    ("Topiramate",    "TX"),
    ("Carbamazepine", "CA"),
    ("Carbamazepine", "FL"),
    ("Carbamazepine", "OH"),
    ("Lacosamide",    "CA"),
    ("Lacosamide",    "FL"),
    ("Lacosamide",    "TX"),
    ("Zonisamide",    "CA"),
    ("Zonisamide",    "FL"),
    ("Valproic Acid", "CA"),
    ("Valproic Acid", "FL"),
    ("Oxcarbazepine", "CA"),
    ("Oxcarbazepine", "FL"),
    ("Perampanel",    "CA"),
    ("Perampanel",    "FL"),
]

print(f"Training {len(ALL_DRUG_STATES)} Prophet models...")
models    = {}
forecasts = {}

for drug, state in ALL_DRUG_STATES:
    print(f"  Training: {drug} — {state}")
    model, forecast = train_prophet(drug, state, df_weekly)
    if model is not None:
        models[(drug, state)]    = model
        forecasts[(drug, state)] = forecast

print(f"\nTrained {len(models)} models successfully.")

---
### Section 7 — Anomaly detection across all drugs

In [ ]:
def detect_anomalies(drug, state, df_actual, forecast, threshold=0.40):
    """Compare actual vs forecast demand. Flag deviations above threshold."""
    actual = (
        df_actual[(df_actual["drug"] == drug) & (df_actual["state"] == state)]
        .rename(columns={"date": "ds"})
        [["ds", "claims", "is_anomaly", "anomaly_name"]]
        .sort_values("ds")
    )
    merged = actual.merge(
        forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
        on="ds", how="left"
    )
    merged["deviation"]     = (merged["claims"] - merged["yhat"]) / merged["yhat"]
    merged["deviation_pct"] = (merged["deviation"] * 100).round(1)
    merged["flagged"]       = (merged["deviation"].abs() > threshold)
    merged["flag_type"]     = merged.apply(
        lambda r: "SPIKE" if r["deviation"] > threshold
                  else "DROP" if r["deviation"] < -threshold
                  else "NORMAL", axis=1
    )
    return merged


print("Running anomaly detection across all drug-state pairs...\n")
detection_results = {}

for drug, state in ALL_DRUG_STATES:
    if (drug, state) in forecasts:
        result = detect_anomalies(drug, state, df_anomaly, forecasts[(drug, state)])
        detection_results[(drug, state)] = result
        flagged = result[result["flagged"]]
        if len(flagged) > 0:
            print(f"{drug} — {state}: {len(flagged)} anomaly weeks detected")
            for _, row in flagged.iterrows():
                print(f"  {row['ds'].date()} | {row['flag_type']} | {row['deviation_pct']:+.1f}%")
        else:
            print(f"{drug} — {state}: no anomalies detected")

In [ ]:
# Visualize detection results — dynamic grid
n     = len(ALL_DRUG_STATES)
cols  = 3
rows  = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes_flat = axes.flatten()

for i, (drug, state) in enumerate(ALL_DRUG_STATES):
    ax = axes_flat[i]
    if (drug, state) not in detection_results:
        ax.set_visible(False)
        continue

    result   = detection_results[(drug, state)]
    forecast = forecasts[(drug, state)]

    ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                    alpha=0.2, color="#2E5F8A", label="95% forecast band")
    ax.plot(forecast["ds"], forecast["yhat"],
            color="#2E5F8A", linewidth=1.5, linestyle="--", label="Forecast")
    ax.plot(result["ds"], result["claims"],
            color="#1e293b", linewidth=1.5, label="Actual")

    spikes = result[result["flag_type"] == "SPIKE"]
    drops  = result[result["flag_type"] == "DROP"]
    if len(spikes) > 0:
        ax.scatter(spikes["ds"], spikes["claims"],
                   color="#E24B4A", zorder=5, s=40, label="Spike")
    if len(drops) > 0:
        ax.scatter(drops["ds"], drops["claims"],
                   color="#EF9F27", zorder=5, s=40, label="Drop")

    ax.set_title(f"{drug} — {state}", fontsize=10)
    ax.set_xlabel("Week")
    ax.set_ylabel("Claims")
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

for j in range(n, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Anomaly Detection — Actual vs Forecast (All Drugs)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("all_drugs_detection_results.png", dpi=150)
plt.show()

---
### Section 8 — Generate plain English alerts via AWS Bedrock

In [ ]:
import os

os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "your-bearer-token-here"

bedrock   = boto3.client("bedrock-runtime", region_name="ap-southeast-2")
LLM_MODEL = "amazon.nova-pro-v1:0"

SYSTEM_PROMPT = """You are a supply chain intelligence assistant at UCB Pharmaceuticals.
Analyze demand anomalies for AED drugs and generate clear, actionable alerts
for the supply chain team. Always include: what is happening, severity level
(High/Medium/Low), likely cause, and recommended action.
Keep the alert under 150 words. Be direct and specific."""


def generate_alert(drug, state, flag_type, deviation_pct,
                   week_date, actual_claims, forecasted_claims):
    prompt = f"""Supply chain anomaly detected:

Drug         : {drug}
State        : {state}
Date         : {week_date}
Anomaly type : {flag_type}
Actual demand: {actual_claims:,} claims
Forecast     : {forecasted_claims:,.0f} claims
Deviation    : {deviation_pct:+.1f}% from expected

Generate a supply chain alert with severity, likely cause, and recommended action."""

    response = bedrock.converse(
        modelId=LLM_MODEL,
        system=[{"text": SYSTEM_PROMPT}],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0}
    )
    return response["output"]["message"]["content"][0]["text"]


print("Bedrock client ready.")

In [ ]:
print("Generating supply chain alerts for all detected anomalies...\n")
print("=" * 70)

all_alerts = []

for (drug, state), result in detection_results.items():
    flagged = result[result["flagged"]]
    for _, row in flagged.iterrows():
        alert_text = generate_alert(
            drug             = drug,
            state            = state,
            flag_type        = row["flag_type"],
            deviation_pct    = row["deviation_pct"],
            week_date        = row["ds"].date(),
            actual_claims    = int(row["claims"]),
            forecasted_claims= row["yhat"],
        )
        all_alerts.append({
            "drug"        : drug,
            "state"       : state,
            "date"        : row["ds"].date(),
            "flag_type"   : row["flag_type"],
            "deviation"   : row["deviation_pct"],
            "actual"      : int(row["claims"]),
            "forecast"    : round(row["yhat"]),
            "alert"       : alert_text,
            "true_anomaly": row["is_anomaly"],
        })

        print(f"ALERT — {drug} / {state} / {row['ds'].date()}")
        print(f"Type: {row['flag_type']} | Deviation: {row['deviation_pct']:+.1f}%")
        print(f"\n{alert_text}")
        print("=" * 70)

print(f"\nTotal alerts generated: {len(all_alerts)}")

---
### Section 9 — Evaluate detection performance

In [ ]:
all_results = []
for (drug, state), result in detection_results.items():
    r = result.copy()
    r["drug_key"]  = drug
    r["state_key"] = state
    all_results.append(r)

df_results = pd.concat(all_results, ignore_index=True)

tp = len(df_results[(df_results["flagged"]==True)  & (df_results["is_anomaly"]==True)])
fp = len(df_results[(df_results["flagged"]==True)  & (df_results["is_anomaly"]==False)])
fn = len(df_results[(df_results["flagged"]==False) & (df_results["is_anomaly"]==True)])
tn = len(df_results[(df_results["flagged"]==False) & (df_results["is_anomaly"]==False)])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("DETECTION PERFORMANCE — ALL DRUGS")
print("=" * 45)
print(f"  True Positives  : {tp:>4}  (correctly caught anomalies)")
print(f"  False Positives : {fp:>4}  (false alarms)")
print(f"  False Negatives : {fn:>4}  (missed anomalies)")
print(f"  True Negatives  : {tn:>4}  (correctly ignored normal weeks)")
print()
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1 Score  : {f1:.3f}")
print()

# Per-drug breakdown
print("PER-DRUG BREAKDOWN")
print("-" * 55)
for drug in sorted(df_results["drug_key"].unique()):
    d     = df_results[df_results["drug_key"] == drug]
    d_tp  = len(d[(d["flagged"]==True)  & (d["is_anomaly"]==True)])
    d_fp  = len(d[(d["flagged"]==True)  & (d["is_anomaly"]==False)])
    d_fn  = len(d[(d["flagged"]==False) & (d["is_anomaly"]==True)])
    d_prec = d_tp / (d_tp + d_fp) if (d_tp + d_fp) > 0 else 0
    d_rec  = d_tp / (d_tp + d_fn) if (d_tp + d_fn) > 0 else 0
    print(f"  {drug:<20} TP:{d_tp} FP:{d_fp} FN:{d_fn} | P:{d_prec:.2f} R:{d_rec:.2f}")

In [ ]:
# Save all outputs
df_alerts = pd.DataFrame(all_alerts)
df_alerts.to_csv("supply_chain_alerts_all_drugs.csv", index=False)
df_results.to_csv("detection_results_all_drugs.csv", index=False)

print("Files saved:")
print("  supply_chain_alerts_all_drugs.csv   — all generated alerts")
print("  detection_results_all_drugs.csv     — full detection results")
print("  all_drugs_weekly_baseline.png       — baseline demand chart")
print("  all_drugs_injected_anomalies.png    — anomaly injection viz")
print("  all_drugs_detection_results.png     — detection results")
print()
print(f"Total alerts generated : {len(df_alerts)}")
print(f"Drugs covered          : {df_alerts['drug'].nunique()}")
print(f"States covered         : {df_alerts['state'].nunique()}")